In [4]:
import pandas as pd
import numpy as np

# Cargamos los datasets ya limpios
sysarmy = pd.read_csv('../data/processed/sysarmy_limpio.csv')
indec   = pd.read_csv('../data/processed/indec_limpio.csv', dtype={'periodo': str})
so_arg  = pd.read_csv('../data/processed/stackoverflow_argentina.csv')
so_latam = pd.read_csv('../data/processed/stackoverflow_latam.csv')

print(f"Sysarmy:           {sysarmy.shape[0]} filas")
print(f"INDEC:             {indec.shape[0]} filas")
print(f"SO Argentina:      {so_arg.shape[0]} filas")
print(f"SO LATAM:          {so_latam.shape[0]} filas")
print("\nDatasets cargados")

Sysarmy:           3676 filas
INDEC:             76 filas
SO Argentina:      149 filas
SO LATAM:          1282 filas

Datasets cargados


In [5]:
# Asignamos el período de la encuesta
PERIODO_ENCUESTA = '202508'  # Agosto 2025

# Buscamos el IPC de ese período
ipc_referencia = indec[indec['periodo'] == PERIODO_ENCUESTA]['indice_ipc'].values[0]
ipc_base = indec[indec['periodo'] == '202001']['indice_ipc'].values[0]

print(f"Período de referencia: {PERIODO_ENCUESTA}")
print(f"IPC agosto 2025:  {ipc_referencia:,.2f}")
print(f"IPC enero 2020:   {ipc_base:,.2f}")
print(f"Inflación acumulada 2020→2025: {((ipc_referencia/ipc_base)-1)*100:.1f}%")

Período de referencia: 202508
IPC agosto 2025:  9,193.24
IPC enero 2020:   289.83
Inflación acumulada 2020→2025: 3071.9%


In [7]:
### Celda 3 — Calcular salario real ajustado por inflación
# El salario real nos dice cuánto vale el sueldo de hoy
# expresado en pesos de enero 2020 (moneda constante)

sysarmy['periodo'] = PERIODO_ENCUESTA
sysarmy['ipc_referencia'] = ipc_referencia
sysarmy['ipc_base'] = ipc_base

# Salario real = salario nominal / (IPC actual / IPC base)
sysarmy['salario_real_2020'] = sysarmy['salario_bruto'] / (ipc_referencia / ipc_base)

print("=== COMPARACIÓN SALARIO NOMINAL VS REAL ===")
print(f"Salario nominal promedio (ARS 2025):  ${sysarmy['salario_bruto'].mean():>15,.0f}")
print(f"Salario real promedio (ARS 2020):     ${sysarmy['salario_real_2020'].mean():>15,.0f}")
print(f"Poder adquisitivo equivalente:        {(sysarmy['salario_real_2020'].mean()/sysarmy['salario_bruto'].mean())*100:.1f}%")

=== COMPARACIÓN SALARIO NOMINAL VS REAL ===
Salario nominal promedio (ARS 2025):  $      3,265,023
Salario real promedio (ARS 2020):     $        102,934
Poder adquisitivo equivalente:        3.2%


In [9]:
### Celda 4 — Agregar tipo de cambio para convertir a USD
# Tipo de cambio oficial aproximado agosto 2025
# Podés actualizarlo si tenés el dato exacto
TC_AGOSTO_2025 = 1342

sysarmy['salario_usd_mensual'] = sysarmy['salario_bruto'] / TC_AGOSTO_2025

print("=== SALARIOS EN USD ===")
print(f"Salario promedio mensual: USD {sysarmy['salario_usd_mensual'].mean():,.0f}")
print(f"Salario mediana mensual:  USD {sysarmy['salario_usd_mensual'].median():,.0f}")
print(f"Salario mínimo:           USD {sysarmy['salario_usd_mensual'].min():,.0f}")
print(f"Salario máximo:           USD {sysarmy['salario_usd_mensual'].max():,.0f}")

=== SALARIOS EN USD ===
Salario promedio mensual: USD 2,433
Salario mediana mensual:  USD 2,161
Salario mínimo:           USD 447
Salario máximo:           USD 7,452


In [10]:
### Celda 5 — Calcular salario anual USD de sysarmy para comparar con SO
sysarmy['salario_usd_anual'] = sysarmy['salario_usd_mensual'] * 12

# Mediana por rol en sysarmy
sysarmy_por_rol = sysarmy.groupby('rol')['salario_usd_anual'].median().reset_index()
sysarmy_por_rol.columns = ['rol', 'salario_usd_anual_sysarmy']
sysarmy_por_rol = sysarmy_por_rol.sort_values('salario_usd_anual_sysarmy', ascending=False)

print("=== TOP 10 ROLES POR SALARIO (sysarmy) ===")
print(sysarmy_por_rol.head(10).to_string(index=False))

=== TOP 10 ROLES POR SALARIO (sysarmy) ===
                 rol  salario_usd_anual_sysarmy
               staff               58122.205663
    jefa del sistema               48926.673621
           architect               42473.919523
        vp / c-level               40493.615499
       riesgos de ti               38450.074516
  manager / director               37555.886736
    technical leader               37495.394933
              varios               35767.511177
ict senior assistant               31296.572280
             infosec               30849.478390


In [11]:
### Celda 6 — Comparar con Stack Overflow LATAM
so_latam_por_rol = so_latam.groupby('rol')['salario_anual_usd'].median().reset_index()
so_latam_por_rol.columns = ['rol', 'salario_usd_anual_latam']

print("=== TOP 10 ROLES POR SALARIO (LATAM - Stack Overflow) ===")
print(so_latam_por_rol.sort_values('salario_usd_anual_latam', ascending=False).head(10).to_string(index=False))

=== TOP 10 ROLES POR SALARIO (LATAM - Stack Overflow) ===
                              rol  salario_usd_anual_latam
Developer, AI apps or physical AI                  89967.5
                  Product manager                  67354.0
              Engineering manager                  63167.0
  DevOps engineer or professional                  50242.0
 Founder, technology or otherwise                  45270.0
                Developer, mobile                  43689.0
 Architect, software or solutions                  43689.0
                    Data engineer                  41834.0
                   AI/ML engineer                  41140.5
            Developer, QA or test                  40794.5


In [12]:
### Celda 7 — Mapeo de roles sysarmy → categorías comparables
mapeo_roles = {
    # Desarrollo
    'developer': 'Developer',
    'desarrollador': 'Developer',
    'fullstack': 'Developer',
    'frontend': 'Developer',
    'backend': 'Developer',
    'mobile': 'Developer, mobile',
    # Data
    'data scientist': 'Data scientist or machine learning specialist',
    'data engineer': 'Data engineer',
    'data analyst': 'Data analyst or scientist',
    'bi': 'Data analyst or scientist',
    # DevOps / Infra
    'devops': 'DevOps engineer or professional',
    'sre': 'DevOps engineer or professional',
    'infraestructura': 'DevOps engineer or professional',
    # QA
    'qa': 'Developer, QA or test',
    'tester': 'Developer, QA or test',
    'quality': 'Developer, QA or test',
    # Management
    'manager': 'Engineering manager',
    'director': 'Engineering manager',
    'technical leader': 'Engineering manager',
    'tech lead': 'Engineering manager',
    'vp': 'Engineering manager',
    # Arquitectura
    'architect': 'Architect, software or solutions',
    # Seguridad
    'infosec': 'Security professional',
    'seguridad': 'Security professional',
}

def mapear_rol(rol):
    if pd.isna(rol):
        return 'Other'
    rol_lower = str(rol).lower()
    for clave, categoria in mapeo_roles.items():
        if clave in rol_lower:
            return categoria
    return 'Other'

sysarmy['rol_estandar'] = sysarmy['rol'].apply(mapear_rol)

print("=== DISTRIBUCIÓN ROLES ESTANDARIZADOS ===")
print(sysarmy['rol_estandar'].value_counts())

=== DISTRIBUCIÓN ROLES ESTANDARIZADOS ===
rol_estandar
Developer                                        1270
Engineering manager                               629
Other                                             614
DevOps engineer or professional                   413
Data analyst or scientist                         200
Developer, QA or test                             188
Data engineer                                     110
Architect, software or solutions                   85
Data scientist or machine learning specialist      85
Security professional                              82
Name: count, dtype: int64


In [13]:
### Celda 8 — Ver qué roles quedaron sin mapear
otros = sysarmy[sysarmy['rol_estandar'] == 'Other']['rol'].value_counts()
print("=== ROLES SIN MAPEAR (top 20) ===")
print(otros.head(20).to_string())

=== ROLES SIN MAPEAR (top 20) ===
rol
ux designer                     83
business analyst                70
recruiter                       63
consultant                      60
helpdesk                        57
networking                      47
designer                        35
functional analyst              34
sales / pre-sales               25
scrum master                    23
dba (database administrator)    19
ux researcher                   14
technical support               11
finance                         10
docente                          9
ux writer                        9
storage / backup                 6
middleware                       6
marketing                        5
noc analyst                      3


In [14]:
### Celda 9 — Mapeo extendido
mapeo_roles_extendido = {
    # Desarrollo
    'developer': 'Developer',
    'desarrollador': 'Developer',
    'fullstack': 'Developer',
    'frontend': 'Developer',
    'backend': 'Developer',
    'programador': 'Developer',
    # Mobile
    'mobile': 'Developer, mobile',
    # Data
    'data scientist': 'Data scientist or machine learning specialist',
    'data engineer': 'Data engineer',
    'data analyst': 'Data analyst or scientist',
    'bi ': 'Data analyst or scientist',
    # DevOps / Infra
    'devops': 'DevOps engineer or professional',
    'sre': 'DevOps engineer or professional',
    'infraestructura': 'DevOps engineer or professional',
    'networking': 'DevOps engineer or professional',
    'storage': 'DevOps engineer or professional',
    'middleware': 'DevOps engineer or professional',
    'noc': 'DevOps engineer or professional',
    # QA
    'qa': 'Developer, QA or test',
    'tester': 'Developer, QA or test',
    'quality': 'Developer, QA or test',
    # Management
    'manager': 'Engineering manager',
    'director': 'Engineering manager',
    'technical leader': 'Engineering manager',
    'tech lead': 'Engineering manager',
    'vp': 'Engineering manager',
    'scrum master': 'Engineering manager',
    # Arquitectura
    'architect': 'Architect, software or solutions',
    # Seguridad
    'infosec': 'Security professional',
    'seguridad': 'Security professional',
    # UX / Diseño
    'ux': 'Designer',
    'designer': 'Designer',
    # Consultoría / Negocio
    'business analyst': 'Business analyst',
    'functional analyst': 'Business analyst',
    'consultant': 'Business analyst',
    'scrum': 'Business analyst',
    # Soporte
    'helpdesk': 'Technical support',
    'technical support': 'Technical support',
    'soporte': 'Technical support',
    # Base de datos
    'dba': 'Database administrator',
    'database': 'Database administrator',
    # RRHH / Recruiting
    'recruiter': 'Other',
    'sales': 'Other',
    'finance': 'Other',
    'marketing': 'Other',
    'docente': 'Other',
}

def mapear_rol_v2(rol):
    if pd.isna(rol):
        return 'Other'
    rol_lower = str(rol).lower()
    for clave, categoria in mapeo_roles_extendido.items():
        if clave in rol_lower:
            return categoria
    return 'Other'

sysarmy['rol_estandar'] = sysarmy['rol'].apply(mapear_rol_v2)

print("=== DISTRIBUCIÓN ROLES ESTANDARIZADOS V2 ===")
print(sysarmy['rol_estandar'].value_counts())
print(f"\nSin mapear (Other): {(sysarmy['rol_estandar'] == 'Other').sum()}")

=== DISTRIBUCIÓN ROLES ESTANDARIZADOS V2 ===
rol_estandar
Developer                                        1270
Engineering manager                               652
DevOps engineer or professional                   475
Data analyst or scientist                         198
Developer, QA or test                             188
Business analyst                                  164
Designer                                          141
Other                                             139
Data engineer                                     110
Architect, software or solutions                   85
Data scientist or machine learning specialist      85
Security professional                              82
Technical support                                  68
Database administrator                             19
Name: count, dtype: int64

Sin mapear (Other): 139


In [16]:
### Celda 10 — Merge sysarmy + Stack Overflow por rol estandarizado
so_latam_por_rol = so_latam.groupby('rol')['salario_anual_usd'].agg(
    salario_usd_anual_latam='median',
    respuestas_latam='count'
).reset_index()
so_latam_por_rol.columns = ['rol_estandar', 'salario_usd_anual_latam', 'respuestas_latam']

# Merge
sysarmy_merged = sysarmy.merge(so_latam_por_rol, on='rol_estandar', how='left')

print(f"Filas antes del merge: {len(sysarmy)}")
print(f"Filas después del merge: {len(sysarmy_merged)}")
print(f"Filas con salario LATAM asignado: {sysarmy_merged['salario_usd_anual_latam'].notna().sum()}")

Filas antes del merge: 3676
Filas después del merge: 3676
Filas con salario LATAM asignado: 1510


In [17]:
### Celda 11 — Calcular brecha salarial vs LATAM
sysarmy_merged['brecha_vs_latam_pct'] = (
    (sysarmy_merged['salario_usd_anual'] - sysarmy_merged['salario_usd_anual_latam'])
    / sysarmy_merged['salario_usd_anual_latam'] * 100
).round(1)

print("=== BRECHA SALARIAL ARGENTINA VS LATAM (por rol) ===")
brecha_por_rol = sysarmy_merged.groupby('rol_estandar').agg(
    salario_arg=('salario_usd_anual', 'median'),
    salario_latam=('salario_usd_anual_latam', 'first'),
    brecha_pct=('brecha_vs_latam_pct', 'median')
).dropna().sort_values('brecha_pct', ascending=True)

print(brecha_por_rol.to_string())

=== BRECHA SALARIAL ARGENTINA VS LATAM (por rol) ===
                                   salario_arg  salario_latam  brecha_pct
rol_estandar                                                             
Developer, QA or test             17731.743666        40794.5      -56.55
DevOps engineer or professional   24143.070045        50242.0      -51.90
Engineering manager               37491.357675        63167.0      -40.65
Data engineer                     28614.008942        41834.0      -31.60
Architect, software or solutions  42473.919523        43689.0       -2.80


In [ ]:
### Celda 12 — Guardar dataset final enriquecido
sysarmy_merged.to_csv('../data/processed/dataset_final.csv', index=False)
print(f"Dataset final guardado: {len(sysarmy_merged)} filas, {sysarmy_merged.shape[1]} columnas")
print(f"\nColumnas nuevas agregadas:")
nuevas = ['rol_estandar', 'salario_usd_mensual', 'salario_usd_anual', 
          'salario_real_2020', 'salario_usd_anual_latam', 'brecha_vs_latam_pct']
for col in nuevas:
    print(f"  + {col}")

✅ Dataset final guardado: 3676 filas, 30 columnas

Columnas nuevas agregadas:
  + rol_estandar
  + salario_usd_mensual
  + salario_usd_anual
  + salario_real_2020
  + salario_usd_anual_latam
  + brecha_vs_latam_pct
